In [1]:
import pandas as pd
import numpy as np
import bioframe as bf

### Overlapping Akita.V2 and AlphaGenome test windows

In [2]:
# MOUSE

akita_path = "/project2/fudenber_735/tensorflow_models/akita/v2/data/mm10/sequences.bed"
alphagenome_path = f"/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/mouse_cell_types/alphagenome_safe_windows.tsv"

In [ ]:
# HUMAN

# akita_path = "/project2/fudenber_735/tensorflow_models/akita/v2/data/hg38/sequences.bed"
# alphagenome_path = f"/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/human_cell_types/alphagenome_safe_windows.tsv"

In [3]:
# reading test data for akita V2
sequences_akita = pd.read_csv(akita_path, sep='\t', names=['chr','start','end','type'])

In [4]:
# reading test data for alpha genome
sequences_test_alpha = pd.read_csv(alphagenome_path, sep='\t')

In [5]:
sequences_test_alpha.rename(columns={'chrom': 'chr'}, inplace=True)

In [6]:
df_overlap = bf.overlap(
        sequences_akita, sequences_test_alpha, how="inner", suffixes=("_akita", "_alpha"), cols1=["chr", "start", "end"], cols2=["chr", "start", "end"],
    )

In [7]:
df_overlap

,chr_akita,start_akita,end_akita,type_akita,chr_alpha,start_alpha,end_alpha,type_alpha
0,chr7,15818752,17129472,fold3,chr7,15841542,16890118,fold0
1,chr7,15818752,17129472,fold3,chr7,15890715,16939291,fold0
2,chr7,15818752,17129472,fold3,chr7,15939888,16988464,fold0
3,chr7,15818752,17129472,fold3,chr7,15989061,17037637,fold0
4,chr7,15818752,17129472,fold3,chr7,16038234,17086810,fold0
...,...,...,...,...,...,...,...,...
128965,chr12,118593536,119904256,fold6,chr12,118387296,119435872,fold0
128966,chr12,118593536,119904256,fold6,chr12,118436469,119485045,fold0
128967,chr12,118593536,119904256,fold6,chr12,118485642,119534218,fold0
128968,chr12,118593536,119904256,fold6,chr12,118534815,119583391,fold0


In [ ]:
# Keep only those Akita windows that overlap AlphaGenome windows all from the same fold,
# and exclude Akita windows that overlap AlphaGenome windows from multiple different folds.

In [8]:
# For each Akita window, check how many *unique* AlphaGenome folds it overlaps
folds_per_akita = (
    df_overlap.groupby(['chr_akita', 'start_akita', 'end_akita', 'type_akita'])['type_alpha']
    .nunique()
    .reset_index(name='n_unique_alpha_folds')
)

In [9]:
# Keep only Akita windows that overlap AlphaGenome windows from exactly one fold
akita_keep = folds_per_akita.query("n_unique_alpha_folds == 1")[['chr_akita', 'start_akita', 'end_akita', 'type_akita']]

In [10]:
# Filter df_overlap accordingly
df_overlap_filtered = df_overlap.merge(
    akita_keep,
    on=['chr_akita', 'start_akita', 'end_akita', 'type_akita'],
    how='inner'
)

In [11]:
# finding the closest windows
df_overlap_filtered["akita_midpoint"] = df_overlap_filtered["start_akita"] + (0.5*(df_overlap_filtered["end_akita"] - df_overlap_filtered["start_akita"]))
df_overlap_filtered["alpha_midpoint"] = df_overlap_filtered["start_alpha"] + (0.5*(df_overlap_filtered["end_alpha"] - df_overlap_filtered["start_alpha"]))
df_overlap_filtered["midpoint_dist"] = np.abs(df_overlap_filtered["akita_midpoint"]-df_overlap_filtered["alpha_midpoint"])

In [12]:
# selecting akita windows with minimal distance from the alpha genome windows
df_sorted = df_overlap_filtered.sort_values(by=['chr_akita', 'start_akita', 'end_akita', 'midpoint_dist'], ascending=[True, True, True, True])
df_unique = df_overlap_filtered.groupby(['chr_akita', 'start_akita', 'end_akita']).first().reset_index()

In [ ]:
df_unique

In [13]:
# renaming columns that are gonna be saved
df_unique = df_unique.rename(columns={"chr_akita" : "chr",
                                     "start_akita" : "start",
                                     "end_akita" : "end"})

In [14]:
# Keep only selected columns
df_to_save = df_unique[['chr', 'start', 'end', 'type_alpha', 'type_akita']]

In [15]:
df_to_save

,chr,start,end,type_alpha,type_akita
0,chr1,3000320,4311040,fold3,fold5
1,chr1,3328000,4638720,fold3,fold5
2,chr1,3655680,4966400,fold3,fold5
3,chr1,3983360,5294080,fold3,fold5
4,chr1,4311040,5621760,fold3,fold5
...,...,...,...,...,...
2935,chrX,167217152,168527872,fold2,fold7
2936,chrX,167544832,168855552,fold2,fold7
2937,chrX,167872512,169183232,fold2,fold7
2938,chrX,168200192,169510912,fold2,fold7


In [ ]:
# for human only
# ORCA's test is chromosome 9 and 10

df_orca_filtered = df_to_save[df_to_save['chr'].isin(['chr9', 'chr10'])].copy()

In [ ]:
df_orca_filtered

In [ ]:
# sampling 500 random windows
df_sampled = df_to_save.sample(n=500, random_state=42)

In [ ]:
df_sampled = df_sampled.reset_index(drop=True)

In [ ]:
df_sampled 

In [ ]:
# Save as TSV (tab-separated, no index)
# df_sampled.to_csv(f"/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/mouse_cell_types/alphagenome_akita_test_overlap_500windows.tsv", sep='\t', index=False)

In [ ]:
df_orca_filtered.to_csv(f"/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/human_cell_types/alphagenome_akita_orca_test_overlap.tsv", sep='\t', index=False)